# TasvirEt Encoder Deneyi -- E5 (MobileCLIP-S2) 16.000 iteration

E1-E4 ile **birebir aynı tarif** (`encoder_experiments.yaml`'in kopyası):
aynı veri, decoder, MLP2 projection, optimizer, stage 1 (250 it), stage 2 =
**16.000 iteration** (checkpoint 4k/8k/12k/16k), aynı seed (42).
Tek fark: deney listesinde yalnızca E5 = MobileCLIP-S2 var ve çıktı ayrı bir klasöre yazılır.

E1-E4 A100'de koşuldu -- bu deney de **A100** ile koşulmalı (Çalışma zamanı → Çalışma zamanı türünü değiştir → A100 GPU).

Sonuçlar: `TRCAP_encoder_exp_s2_16k/` (E1-E4'ün `TRCAP_encoder_exp/` ve 12k E5'in `TRCAP_encoder_exp_extra/` klasörlerine dokunulmaz).


## 1. Ortam

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert 'A100' in torch.cuda.get_device_name(0), 'A100 değil: ' + torch.cuda.get_device_name(0)

In [ ]:
%cd /content
!rm -rf /content/TRCAP
!git clone -b encoder-exp https://github.com/mhr871/TRCAP.git
%cd /content/TRCAP
!git rev-parse --short HEAD

In [ ]:
!python -m pip install -r requirements_colab.txt
!python -m pip install --no-deps git+https://github.com/apple/ml-mobileclip.git
!python -c "import torch, transformers, timm, open_clip, mobileclip; print('torch', torch.__version__, '| transformers', transformers.__version__, '| timm', timm.__version__, '| open_clip', open_clip.__version__, '| cuda', torch.cuda.is_available())"

## 2. Veri ve MobileCLIP-S2 pretrained ağırlığı

In [ ]:
!python tools/prepare_tasviret.py --allow-missing-images
!python tools/download_tasviret_images.py
!python tools/prepare_tasviret.py --images-root Data/flickr8k/images

In [ ]:
!python tools/download_mobileclip.py --model mobileclip_s2

## 3. Preflight (eğitim başlatmaz)
`PREFLIGHT PASSED` görmeden 4. adıma geçmeyin.

In [ ]:
!PYTHONPATH=/content/TRCAP python run_encoder_experiments.py   --plan configs/encoder_exp/encoder_experiments_s2_16k.yaml --preflight

## 4. E5 eğitimi (16k)
Çıktı: `/content/drive/MyDrive/TRCAP_encoder_exp_s2_16k/E5_MobileCLIP_S2/{checkpoints,logs,metrics,config}`.

In [ ]:
!PYTHONPATH=/content/TRCAP python -u run_encoder_experiments.py   --plan configs/encoder_exp/encoder_experiments_s2_16k.yaml   --save-dir /content/drive/MyDrive/TRCAP_encoder_exp_s2_16k

## 5. Sonuç

In [ ]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/TRCAP_encoder_exp_s2_16k/encoder_results.csv')
cols = ['experiment_id', 'encoder', 'status', 'gpu', 'git_commit', 'max_iter', 'best_iter', 'best_val_CIDEr',
        'test_best_CIDEr', 'test_best_Bleu_4', 'test_best_ROUGE_L',
        'test_final_CIDEr', 'test_final_Bleu_4', 'test_final_ROUGE_L', 'training_time']
df[cols].T